In [23]:
from langchain_cohere import CohereRerank
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [24]:
reranker = CohereRerank(
    model="rerank-v3.5",
    top_n=5
)

In [25]:
query = "What is the annual leave policy?"

from langchain_core.documents import Document

docs = [
    Document(page_content="The company provides medical insurance to all full-time employees."),
    Document(page_content="Employees are entitled to 20 days of annual leave every calendar year."),  # Relevant
    Document(page_content="Employees must provide a medical certificate for sick leave longer than two days."),
    Document(page_content="Annual performance reviews are conducted every December."),
    Document(page_content="Annual leave requests must be submitted through the HR portal."),  # Relevant
    Document(page_content="Employees receive their salary on the last working day of each month."),
    Document(page_content="New employees have a three-month probation period."),
    Document(page_content="The company provides transportation for employees working late."),
    Document(page_content="Employees can work remotely two days per week."),
    Document(page_content="Overtime compensation is calculated according to company policy.")
]

In [26]:
results = reranker.compress_documents(
    documents=docs,
    query=query
)

In [27]:
for i, doc in enumerate(results, 1):
    print(f"Rank {i}")
    print(doc.page_content)
    print()

Rank 1
Employees are entitled to 20 days of annual leave every calendar year.

Rank 2
Annual leave requests must be submitted through the HR portal.

Rank 3
Employees receive their salary on the last working day of each month.

Rank 4
Overtime compensation is calculated according to company policy.

Rank 5
Employees must provide a medical certificate for sick leave longer than two days.



In [43]:
def precision_at_k(retrieved, relevant, k):
    top_k = retrieved[:k]

    relevant_count = 0

    for doc in top_k:
        if doc in relevant:
            relevant_count += 1

    return relevant_count / k
    
def recall_at_k(retrieved, relevant, k):
    top_k = retrieved[:k]

    found = 0

    for doc in top_k:
        if doc in relevant:
            found += 1

    return found / len(relevant)
    

def mrr(retrieved, relevant):
    for position, doc in enumerate(retrieved, start=1):

        if doc in relevant:
            return 1 / position

    return 0


import math
def ndcg_at_k(retrieved, relevant, k):
    top_k = retrieved[:k]

    score = 0

    for position, doc in enumerate(top_k, start=1):

        if doc in relevant:
            score += 1 / math.log2(position + 1)

    ideal_score = 0

    for position in range(1, min(len(relevant), k) + 1):
        ideal_score += 1 / math.log2(position + 1)

    return score / ideal_score

def hit_rate_at_k(retrieved, relevant, k):
    top_k = retrieved[:k]

    for doc in top_k:
        if doc in relevant:
            return 1

    return 0

In [44]:
retrieved_indices = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

relevant_indices = {1, 4}

In [45]:
print("Precision@5:", precision_at_k(retrieved_indices, relevant_indices, 5))
print("Recall@5:", recall_at_k(retrieved_indices, relevant_indices, 5))
print("Hit Rate@5:", hit_rate_at_k(retrieved_indices, relevant_indices, 5))
print("MRR:", mrr(retrieved_indices, relevant_indices))
print("NDCG@5:", ndcg_at_k(retrieved_indices, relevant_indices, 5))

Precision@5: 0.4
Recall@5: 1.0
Hit Rate@5: 1
MRR: 0.5
NDCG@5: 0.6240505200038379
